# Final subsample search

This notebook implements the targeted post-midterm extension. The goal is not to run a broad specification garden. It tests a small number of theoretically motivated subsamples where birth-region favoritism should be most likely to survive: geography, durable political power, luminosity headroom, and pollution-support samples.

Decision rule: run the nightlights first stage first. A subsample is treated as a credible follow-through candidate only if the post-2005 first stage is positive, at least 0.005, has p < 0.10, and has at least 50 treated observations. If no subsample clears that bar, the notebook reports the strongest positive diagnostic samples separately and does not interpret their pollution estimates as confirmatory evidence.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

warnings.filterwarnings("ignore")

ROOT = Path("..")
DATA = ROOT / "data"
SCRIPTS = (ROOT / "scripts").resolve()
if str(SCRIPTS) not in sys.path:
    sys.path.append(str(SCRIPTS))

from gid_utils import valid_gid_mask

OUT_FIRST = ROOT / "analysis" / "final_subsample_first_stage.csv"
OUT_POLL = ROOT / "analysis" / "final_subsample_pollution_followthrough.csv"
OUT_FIG = ROOT / "analysis" / "final_subsample_first_stage_plot.png"

SSA = set("AGO BDI BEN BFA BWA CAF CIV CMR COD COG COM CPV DJI ERI ETH GAB GHA GIN GMB GNB GNQ KEN LBR LSO MDG MLI MOZ MRT MUS MWI NAM NER NGA RWA SDN SEN SLE SOM SSD STP SWZ TCD TGO TZA UGA ZAF ZMB ZWE".split())

def load_plad():
    plad = pd.read_stata(DATA / "political leaders" / "PLAD_April_2024.dta")
    plad = plad[plad["foreign_leader"] == "0"].copy()
    birth_gid = "gid_2" if "gid_2" in plad.columns else "gid_1"
    plad = plad.loc[valid_gid_mask(plad[birth_gid])].copy()
    plad["startyear"] = plad["startyear"].astype(int)
    plad["endyear"] = plad["endyear"].astype(int)
    plad = plad[plad["archigos_id"].astype(str).str.strip() != "."].copy()
    plad = plad.sort_values(["gid_0", "startyear"]).reset_index(drop=True)

    fixed = plad.copy().reset_index(drop=True)
    for gid_0, group in fixed.groupby("gid_0"):
        idxs = group.index.tolist()
        for i in range(len(idxs) - 1):
            curr, nxt = idxs[i], idxs[i + 1]
            if fixed.loc[curr, "endyear"] >= fixed.loc[nxt, "startyear"]:
                fixed.loc[curr, "endyear"] = fixed.loc[nxt, "startyear"] - 1
    fixed["spell_length_total"] = (fixed["endyear"] - fixed["startyear"] + 1).clip(lower=0)
    return fixed, birth_gid

PLAD_FIXED, BIRTH_GID = load_plad()
COUNTRY_CONTINENT = (
    PLAD_FIXED[["gid_0", "continent"]]
    .dropna()
    .drop_duplicates("gid_0")
    .rename(columns={"gid_0": "GID_0"})
)

def early_support(panel, value_col, years):
    early = panel[panel["year"].isin(years)].dropna(subset=[value_col]).copy()
    country_mean = early.groupby("GID_0")[value_col].mean().rename(f"baseline_{value_col}").reset_index()
    region_mean = early.groupby(["GID_0", "GID_2"])[value_col].mean().rename("region_mean").reset_index()
    country_sd = region_mean.groupby("GID_0")["region_mean"].std().rename(f"dispersion_{value_col}").reset_index()
    out = country_mean.merge(country_sd, on="GID_0", how="outer")
    for col in [f"baseline_{value_col}", f"dispersion_{value_col}"]:
        out[f"high_{col}"] = out[col] >= out[col].median()
    return out

NO2_RAW = pd.read_parquet(DATA / "no2_adm2_acag_panel.parquet")
NO2_RAW = NO2_RAW.loc[valid_gid_mask(NO2_RAW["GID_2"])].dropna(subset=["no2_mean"]).copy()
NO2_SUPPORT = early_support(NO2_RAW, "no2_mean", range(2005, 2008))

PM25_RAW = pd.read_parquet(DATA / "pm25_adm2_acag_panel.parquet")
PM25_RAW = PM25_RAW.loc[valid_gid_mask(PM25_RAW["GID_2"])].dropna(subset=["pm25_mean"]).copy()
PM25_RAW = PM25_RAW[(PM25_RAW["pm25_mean"] >= 0) & (PM25_RAW["pm25_mean"] < 1000)].copy()
PM25_SUPPORT = early_support(PM25_RAW, "pm25_mean", range(1998, 2001))

def load_vdem(start, end):
    vdem = pd.read_csv(DATA / "vdem" / "V-Dem-CY-Core-v15.csv", usecols=["country_text_id", "year", "v2x_polyarchy"])
    vdem = vdem.rename(columns={"country_text_id": "GID_0", "v2x_polyarchy": "democracy"})
    vdem = vdem.dropna(subset=["democracy"])
    return vdem[(vdem["year"] >= start) & (vdem["year"] <= end)].copy()

print(f"PLAD leaders after filters: {len(PLAD_FIXED)} | birth column: {BIRTH_GID}")

In [ ]:
def build_treatment(base, start, end):
    rows = []
    spell_rows = []
    base_small = base[["GID_2", "GID_0", "year", "ntl_mean"]].copy()
    base_small["ntl_rank_country_year"] = base_small.groupby(["GID_0", "year"])["ntl_mean"].rank(pct=True)

    for idx, row in PLAD_FIXED.iterrows():
        spell_start = max(int(row["startyear"]), start)
        spell_end = min(int(row["endyear"]), end)
        if spell_start > spell_end:
            continue

        entry = base_small[(base_small["GID_2"] == row[BIRTH_GID]) & (base_small["year"] == spell_start)]
        birth_dn = float(entry["ntl_mean"].iloc[0]) if len(entry) else np.nan
        birth_rank = float(entry["ntl_rank_country_year"].iloc[0]) if len(entry) else np.nan
        spell_rows.append({
            "spell_id": idx,
            "GID_0": row["gid_0"],
            "birth_gid": row[BIRTH_GID],
            "spell_length_total": row["spell_length_total"],
            "observed_years_in_window": spell_end - spell_start + 1,
            "birth_entry_dn": birth_dn,
            "birth_entry_rank": birth_rank,
        })
        for year in range(spell_start, spell_end + 1):
            rows.append({
                "GID_2": row[BIRTH_GID],
                "GID_0": row["gid_0"],
                "year": year,
                "spell_id": idx,
                "spell_length_total": row["spell_length_total"],
                "observed_years_in_window": spell_end - spell_start + 1,
                "birth_entry_dn": birth_dn,
                "birth_entry_rank": birth_rank,
            })

    leader_years = pd.DataFrame(rows)
    leader_years = leader_years.loc[valid_gid_mask(leader_years["GID_2"])].copy()
    if BIRTH_GID == "gid_1":
        adm1_to_adm2 = base[["GID_2", "GID_1"]].drop_duplicates()
        leader_years = (
            leader_years.rename(columns={"GID_2": "GID_1"})
            .merge(adm1_to_adm2, on="GID_1", how="inner")
            [["GID_2", "GID_0", "year", "spell_id", "spell_length_total", "observed_years_in_window", "birth_entry_dn", "birth_entry_rank"]]
        )
    leader_years = leader_years.drop_duplicates(subset=["GID_2", "year"])
    leader_years["birth_region_leader"] = 1
    spell_map = leader_years[["GID_0", "year", "spell_id", "spell_length_total", "observed_years_in_window", "birth_entry_dn", "birth_entry_rank"]].drop_duplicates(subset=["GID_0", "year"])
    spell_info = pd.DataFrame(spell_rows).drop_duplicates("spell_id")
    return leader_years, spell_map, spell_info

def build_ntl_panel(ntl_path, start, end):
    ntl = pd.read_parquet(ntl_path)
    ntl = ntl[(ntl["year"] >= start) & (ntl["year"] <= end)].copy()
    ntl = ntl.loc[valid_gid_mask(ntl["GID_2"])].dropna(subset=["ntl_mean"]).copy()
    leader_years, spell_map, spell_info = build_treatment(ntl, start, end)

    panel = ntl.merge(leader_years[["GID_2", "year", "birth_region_leader"]], on=["GID_2", "year"], how="left")
    panel["birth_region_leader"] = panel["birth_region_leader"].fillna(0).astype(int)
    panel = panel.merge(spell_map, on=["GID_0", "year"], how="left")
    panel["spell_id"] = panel["spell_id"].fillna(panel["GID_0"] + "_" + panel["year"].astype(str) + "_noleader").astype(str)
    panel = panel.sort_values(["GID_2", "year"]).copy()
    panel["brl_lag1"] = panel.groupby("GID_2")["birth_region_leader"].shift(1).fillna(0).astype(int)
    panel["brl_lag2"] = panel.groupby("GID_2")["birth_region_leader"].shift(2).fillna(0).astype(int)
    panel["ln_ntl"] = np.log(panel["ntl_mean"] + 0.01)
    panel["country_year"] = panel["GID_0"] + "_" + panel["year"].astype(str)
    panel["ntl_rank_country_year"] = panel.groupby(["GID_0", "year"])["ntl_mean"].rank(pct=True)
    panel = panel.merge(COUNTRY_CONTINENT, on="GID_0", how="left")
    panel["is_ssa"] = panel["GID_0"].isin(SSA)
    panel = panel.merge(load_vdem(start, end), on=["GID_0", "year"], how="left")
    panel = panel.merge(NO2_SUPPORT, on="GID_0", how="left")
    panel = panel.merge(PM25_SUPPORT, on="GID_0", how="left")
    return panel, spell_info

def run_fe(panel, outcome, treatment="birth_region_leader", sample="Pooled", window="", mask=None):
    d = panel if mask is None else panel.loc[mask(panel)].copy()
    d = d.dropna(subset=[outcome, "country_year", "spell_id", treatment]).copy()
    if d.empty or d[treatment].sum() == 0 or d["GID_0"].nunique() < 2:
        return {"sample": sample, "window": window, "outcome": outcome, "coef": np.nan, "se": np.nan, "pval": np.nan, "nobs": len(d), "treated": int(d[treatment].sum()) if len(d) else 0, "clusters": d["spell_id"].nunique() if len(d) else 0, "countries": d["GID_0"].nunique() if len(d) else 0, "error": "insufficient support"}
    try:
        idx = d.set_index(["GID_2", "year"])
        model = PanelOLS.from_formula(f"{outcome} ~ {treatment} + EntityEffects", data=idx, other_effects=idx["country_year"], drop_absorbed=True)
        res = model.fit(cov_type="clustered", clusters=idx["spell_id"])
        return {"sample": sample, "window": window, "outcome": outcome, "coef": float(res.params[treatment]), "se": float(res.std_errors[treatment]), "pval": float(res.pvalues[treatment]), "nobs": int(res.nobs), "treated": int(d[treatment].sum()), "clusters": int(d["spell_id"].nunique()), "countries": int(d["GID_0"].nunique()), "error": ""}
    except Exception as exc:
        return {"sample": sample, "window": window, "outcome": outcome, "coef": np.nan, "se": np.nan, "pval": np.nan, "nobs": len(d), "treated": int(d[treatment].sum()), "clusters": int(d["spell_id"].nunique()), "countries": int(d["GID_0"].nunique()), "error": str(exc)[:180]}

MASKS = [
    ("Pooled", lambda d: pd.Series(True, index=d.index)),
    ("Africa", lambda d: d["continent"].eq("Africa")),
    ("Non-Africa", lambda d: ~d["continent"].eq("Africa")),
    ("Asia", lambda d: d["continent"].eq("Asia")),
    ("Non-Asia", lambda d: ~d["continent"].eq("Asia")),
    ("Sub-Saharan Africa", lambda d: d["is_ssa"].fillna(False)),
    ("Non-SSA", lambda d: ~d["is_ssa"].fillna(False)),
    ("Long-tenure leaders", lambda d: d["spell_length_total"] >= 5),
    ("Short-tenure leaders", lambda d: d["spell_length_total"] < 5),
    ("Long-tenure autocrats", lambda d: (d["spell_length_total"] >= 5) & (d["democracy"] < 0.3)),
    ("Hybrid regimes", lambda d: (d["democracy"] >= 0.3) & (d["democracy"] < 0.7)),
    ("Democracies", lambda d: d["democracy"] >= 0.7),
    ("Drop top 10% luminosity", lambda d: d["ntl_rank_country_year"] <= 0.90),
    ("Drop top 5% luminosity", lambda d: d["ntl_rank_country_year"] <= 0.95),
    ("Leader birth DN < 61", lambda d: d["birth_entry_dn"].isna() | (d["birth_entry_dn"] < 61)),
    ("Low birth brightness", lambda d: d["birth_entry_rank"] <= d["birth_entry_rank"].median()),
    ("High birth brightness", lambda d: d["birth_entry_rank"] > d["birth_entry_rank"].median()),
    ("High baseline NO2 countries", lambda d: d["high_baseline_no2_mean"].fillna(False)),
    ("Low baseline NO2 countries", lambda d: ~d["high_baseline_no2_mean"].fillna(False)),
    ("High NO2 dispersion countries", lambda d: d["high_dispersion_no2_mean"].fillna(False)),
    ("Low NO2 dispersion countries", lambda d: ~d["high_dispersion_no2_mean"].fillna(False)),
]

In [ ]:
dmsp_1992, _ = build_ntl_panel(DATA / "nightlights_adm2_panel.parquet", 1992, 2013)
dmsp_2005, _ = build_ntl_panel(DATA / "nightlights_adm2_panel.parquet", 2005, 2013)
harm_2005, _ = build_ntl_panel(DATA / "nightlights_adm2_green_favoritism_panel.parquet", 2005, 2019)

rows = []
for name, mask in [("Pooled", MASKS[0][1]), ("Africa", MASKS[1][1]), ("Sub-Saharan Africa", MASKS[5][1]), ("Non-SSA", MASKS[6][1])]:
    rows.append(run_fe(dmsp_1992, "ln_ntl", sample=name, window="DMSP 1992-2013", mask=mask))
for name, mask in MASKS:
    rows.append(run_fe(dmsp_2005, "ln_ntl", sample=name, window="DMSP 2005-2013", mask=mask))
for name, mask in MASKS:
    rows.append(run_fe(harm_2005, "ln_ntl", sample=name, window="Harmonized 2005-2019", mask=mask))

first_stage = pd.DataFrame(rows)
first_stage["ci_low"] = first_stage["coef"] - 1.96 * first_stage["se"]
first_stage["ci_high"] = first_stage["coef"] + 1.96 * first_stage["se"]
first_stage.to_csv(OUT_FIRST, index=False)

first_stage.sort_values(["window", "sample"])[["sample", "window", "coef", "se", "pval", "nobs", "treated", "clusters", "countries"]]

In [ ]:
post = first_stage[first_stage["window"].isin(["DMSP 2005-2013", "Harmonized 2005-2019"])].copy()
credible = post[(post["coef"] >= 0.005) & (post["pval"] < 0.10) & (post["treated"] >= 50)].copy()
if credible.empty:
    credible = post[(post["coef"] >= 0.005) & (post["treated"] >= 50)].sort_values("coef", ascending=False).head(3).copy()
    credible["credible_rule"] = "diagnostic_top_positive_not_statistically_credible"
else:
    credible["credible_rule"] = "coef>=0.005 and p<0.10"

mask_map = dict(MASKS)
no2 = NO2_RAW[(NO2_RAW["year"] >= 2005) & (NO2_RAW["year"] <= 2019)][["GID_2", "year", "no2_mean"]].copy()
poll_no2 = harm_2005.merge(no2, on=["GID_2", "year"], how="inner").dropna(subset=["no2_mean"]).copy()
poll_no2 = poll_no2[poll_no2["no2_mean"] >= 0].copy()
poll_no2["ln_no2"] = np.log(poll_no2["no2_mean"] + 0.01)
poll_no2["no2_pollution_intensity"] = poll_no2["ln_no2"] - poll_no2["ln_ntl"]

pm25 = PM25_RAW[(PM25_RAW["year"] >= 2005) & (PM25_RAW["year"] <= 2019)][["GID_2", "year", "pm25_mean"]].copy()
poll_pm25 = harm_2005.merge(pm25, on=["GID_2", "year"], how="inner").dropna(subset=["pm25_mean"]).copy()
poll_pm25["ln_pm25"] = np.log(poll_pm25["pm25_mean"] + 0.01)
poll_pm25["pm25_pollution_intensity"] = poll_pm25["ln_pm25"] - poll_pm25["ln_ntl"]

poll_rows = []
for sample in dict.fromkeys(credible["sample"].tolist()):
    mask = mask_map[sample]
    base = credible[credible["sample"].eq(sample)].iloc[0]
    for outcome, panel, label in [
        ("ln_ntl", poll_no2, "Nightlights (NO2-matched)"),
        ("ln_no2", poll_no2, "NO2"),
        ("no2_pollution_intensity", poll_no2, "NO2 pollution intensity"),
        ("ln_pm25", poll_pm25, "PM2.5"),
        ("pm25_pollution_intensity", poll_pm25, "PM2.5 pollution intensity"),
    ]:
        result = run_fe(panel, outcome, sample=sample, window="2005-2019 pollution window", mask=mask)
        result["outcome_label"] = label
        result["first_stage_window"] = base["window"]
        result["first_stage_coef"] = base["coef"]
        result["first_stage_pval"] = base["pval"]
        result["credible_rule"] = base["credible_rule"]
        poll_rows.append(result)

pollution_followthrough = pd.DataFrame(poll_rows)
pollution_followthrough.to_csv(OUT_POLL, index=False)

plot_samples = ["Pooled", "Africa", "Sub-Saharan Africa", "Long-tenure leaders", "Long-tenure autocrats", "Drop top 10% luminosity", "Leader birth DN < 61", "Low birth brightness", "High baseline NO2 countries"]
plot_df = first_stage[((first_stage["window"] == "DMSP 2005-2013") & (first_stage["sample"].isin(plot_samples))) | ((first_stage["window"] == "Harmonized 2005-2019") & (first_stage["sample"] == "Pooled"))].copy()
plot_df["label"] = plot_df["sample"] + " (" + plot_df["window"].str.replace("Harmonized ", "harm. ", regex=False).str.replace("DMSP ", "", regex=False) + ")"
plot_df = plot_df.sort_values("coef")

fig, ax = plt.subplots(figsize=(8, 5.2))
y = np.arange(len(plot_df))
ax.errorbar(plot_df["coef"], y, xerr=1.96 * plot_df["se"], fmt="o", color="#1f4e5f", ecolor="#8aa6b1", capsize=3)
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["label"])
ax.set_xlabel("Birth-region coefficient on log nightlights (lag 0)")
ax.set_title("Targeted subsample first-stage estimates")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_FIG, dpi=200, bbox_inches="tight")

credible[["sample", "window", "coef", "se", "pval", "treated", "credible_rule"]], pollution_followthrough[["sample", "outcome_label", "coef", "se", "pval", "nobs", "first_stage_window", "first_stage_coef", "first_stage_pval", "credible_rule"]]

## Interpretation

The strongest new result is historical rather than a post-2005 rescue: Africa and Sub-Saharan Africa have large positive first-stage coefficients in the full DMSP 1992-2013 window. In the post-2005 windows, however, the targeted subsample search does not produce a statistically credible first stage under the pre-specified rule.

The largest later-window diagnostic estimates are in Africa, Sub-Saharan Africa, and low-NO2-dispersion countries in the harmonized 2005-2019 panel. These are positive and larger than the pooled null, but imprecise. The pollution follow-through table is therefore diagnostic only and should not be presented as evidence for green favoritism.